# House Prices — Advanced Regression Techniques

Cleaned version of `notebooks_Untitled.ipynb`. Same model family (one-hot + XGBoost grid, plus a LassoCV baseline), repaired so the numbers are valid.

**Flow:** load → split columns → preprocess → baseline → tuned XGBoost → submission.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

DATA_DIR = Path.home() / "Downloads/house-prices-advanced-regression-techniques"
RANDOM_STATE = 42
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. Load

`y` stays on the price scale here; the log transform happens once, explicitly, right before fitting.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

y = train.pop("SalePrice")
test_ids = test.pop("Id")
X = train.drop(columns=["Id"])

y_log = np.log1p(y)
print(X.shape, test.shape, round(float(y_log.mean()), 3))

## 3. Column types

Note the `dtype.kind` check: under pandas 3 strings are dtype `str`, so `select_dtypes(include=['object'])` returns an **empty list** and every categorical silently lands in the numeric pipeline.

In [ ]:
cat_cols = [c for c in X.columns if X[c].dtype.kind in "OUS"]
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"{len(cat_cols)} categorical, {len(num_cols)} numeric")

## 4. Preprocessing

- categorical → fill with `"None"` (a real level in this dataset, not missing) → **one-hot**
- numeric → median impute → standardise

One-hot rather than `OrdinalEncoder`: the latter invents an ordering, which is meaningless for `Neighborhood` and actively harmful for the linear model.

In [ ]:
def make_preprocessor(cat_cols, num_cols, scale_numeric=True):
    cat_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="None")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    num_steps = [("impute", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scale", StandardScaler()))
    return ColumnTransformer([
        ("cat", cat_pipe, cat_cols),
        ("num", Pipeline(num_steps), num_cols),
    ])

preprocessor = make_preprocessor(cat_cols, num_cols)

## 5. Baseline — LassoCV

Scores are **RMSE on `log1p`**, which is exactly the competition's RMSLE. `neg_root_mean_squared_log_error` would take `log1p` a second time.

In [ ]:
lasso = Pipeline([("prep", make_preprocessor(cat_cols, num_cols)),
                   ("model", LassoCV(cv=5, max_iter=50000, random_state=RANDOM_STATE, n_jobs=-1))])
scores = cross_val_score(lasso, X, y_log, cv=CV, scoring="neg_root_mean_squared_error", n_jobs=-1)
print(f"lasso CV RMSLE {-scores.mean():.5f} +/- {scores.std():.5f}")

## 6. XGBoost with a grid

The step is named `model`, so every key is `model__*`. The original used `rf__*` while holding an XGBoost — it runs, but it misleads whoever reads it next.

In [ ]:
param_grid = {
    "model__max_depth": [3, 4, 5],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__n_estimators": [300, 600],
    "model__subsample": [0.8],
    "model__colsample_bytree": [0.8],
}

xgb_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", XGBRegressor(objective="reg:squarederror", tree_method="hist",
                           random_state=RANDOM_STATE, n_jobs=-1)),
])

grid = GridSearchCV(xgb_pipe, param_grid, cv=CV,
                    scoring="neg_root_mean_squared_error", n_jobs=-1, verbose=1)
grid.fit(X, y_log)
print(grid.best_params_)
print(f"xgb CV RMSLE {-grid.best_score_:.5f}")

## 7. Submission

`expm1` is not optional — the model predicts in log space, Kaggle expects dollars.

In [ ]:
y_pred = np.expm1(grid.best_estimator_.predict(test))
submission = pd.DataFrame({"Id": test_ids, "SalePrice": np.clip(y_pred, 30_000, 800_000)})
submission.to_csv("submission.csv", index=False)
print(submission.shape, round(float(submission.SalePrice.median())))

## 8. Where the score actually goes

This is the original method repaired: **CV 0.126**, consistent with the 0.14548 leaderboard once you allow for the usual optimism. The step to ~0.105 in `train_pipeline.py` comes from feature engineering (ordinal quality scales, `TotalSF`, basement/garage semantics, `LotFrontage` by neighbourhood) and stacking — not from a bigger grid.